<a href="https://colab.research.google.com/github/yulimmm/web-crawling/blob/getReview/GetReviews_iOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install xmltodict

In [4]:
pip install pandas

In [5]:
pip install requests

In [ ]:
import requests
import xmltodict
import pandas as pd
import os

# 텍스트만 추출하는 함수
def extract_text_content(content):
    if isinstance(content, list):
        for item in content:
            if item.get('@type') == 'text':
                return item.get('#text', '').strip()
    elif isinstance(content, dict):
        return content.get('#text', '').strip()
    return str(content).strip()

# 리뷰 수집 함수
def get_ios_reviews_all(app_id, country):
    url = f'https://itunes.apple.com/{country}/rss/customerreviews/page=1/id={app_id}/sortBy=mostRecent/xml'
    response = requests.get(url)
    xml_str = response.content.decode('utf-8') #csv 최적화 글자
    data = xmltodict.parse(xml_str)

    entries = data['feed'].get('entry', [])
    if isinstance(entries, dict):  # 리뷰가 1개인 경우
        entries = [entries]

    reviews = []
    for entry in entries:
        if 'author' not in entry:  # 앱 설명 entry는 제외
            continue
        reviews.append({
            '작성자': entry['author']['name'],
            '별점': int(entry['im:rating']),
            '제목': entry['title'],
            '내용': extract_text_content(entry['content']),
            '작성일': entry['updated'],
            '앱버전': entry.get('im:version', '')
        })

    # DataFrame 생성
    df = pd.DataFrame(reviews)
    df['작성일'] = pd.to_datetime(df['작성일'])

    # 파일 저장
    csv_filename = f'app_reviews_{app_id}_{country}.csv'
    df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

    print(f"✅ CSV 저장 완료: {os.path.abspath(csv_filename)}")
    return df

# 앱 ID와 국가 설정
country = 'kr'
ios_app_id = '1469506430' # 앱에따라 ID 수정

# 실행
df_reviews = get_ios_reviews_all(ios_app_id, country)

# 미리보기 출력
print(df_reviews.head())


✅ CSV 저장 완료: /content/app_reviews_1469506430_kr.csv
          작성자  별점                 제목  \
0         구방니   5        영원히썸원이열리지않음   
1         캥사이   2      광고 너무한 거 아닌가요   
2       sksbs   5     재밌어요 아쉬운점도 있고!   
3        마크대지   2  썸로그 좀 고쳐주세요 ㅠㅠㅠㅠㅠ   
4  Herrrrrr07   5                  .   

                                                  내용  \
0      백만년을해도열리지않음난영원히섬원을못하고영원히친구핰ㄴ테욕을머금도와주세요썸원안열려미친   
1  아니 100자 쓰는 것만 광고 보게 하더니 요즘은 안 넘어도 광고가 2개씩 나와요;...   
2                 질문팩에 질문들이 더 여러가지가 업데이트되었으면 좋겠어요 ㅋㅋ   
3  썸로그 업데이트 됐는데 이상해졌어요 \n지금까지 써놓은 썸로그를 우리함께로 한적이 ...   
4                     비록 헤어지긴 했지만 많은 추억들 만들어줘서 감사합니다   

                        작성일    앱버전  
0 2025-06-07 06:49:23-07:00  2.2.0  
1 2025-06-07 06:10:44-07:00  2.2.0  
2 2025-06-07 00:36:22-07:00  2.2.0  
3 2025-06-06 08:44:56-07:00  2.2.0  
4 2025-06-06 07:47:16-07:00  2.2.0  


In [ ]:
import requests
import xmltodict
import pandas as pd
import os
import time

# 텍스트만 추출하는 함수
def extract_text_content(content):
    if isinstance(content, list):
        for item in content:
            if item.get('@type') == 'text':
                return item.get('#text', '').strip()
    elif isinstance(content, dict):
        return content.get('#text', '').strip()
    return str(content).strip()

# 여러 페이지에서 리뷰 수집
def get_ios_reviews_all(app_id, country, max_page=10):
    reviews = []

    for page in range(1, max_page + 1):
        url = f'https://itunes.apple.com/{country}/rss/customerreviews/page={page}/id={app_id}/sortBy=mostRecent/xml'
        response = requests.get(url)
        if response.status_code != 200:
            print(f"⚠️ 페이지 {page} 요청 실패 (status code: {response.status_code})")
            break

        xml_str = response.content.decode('utf-8')
        try:
            data = xmltodict.parse(xml_str)
        except Exception as e:
            print(f"⚠️ XML 파싱 실패 (page {page}): {e}")
            break

        entries = data['feed'].get('entry', [])
        if isinstance(entries, dict):
            entries = [entries]

        if len(entries) <= 1:  # 리뷰가 없거나 앱 정보만 있을 경우 종료
            break

        for entry in entries:
            if 'author' not in entry:
                continue
            reviews.append({
                '작성자': entry['author']['name'],
                '별점': int(entry['im:rating']),
                '제목': entry['title'],
                '내용': extract_text_content(entry['content']),
                '작성일': entry['updated'],
                '앱버전': entry.get('im:version', '')
            })

        time.sleep(0.5)  # 서버에 무리 안 가게 대기

    # DataFrame 생성 및 저장
    df = pd.DataFrame(reviews)
    df['작성일'] = pd.to_datetime(df['작성일'])

    csv_filename = f'app_reviews_{app_id}_{country}.csv'
    df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

    print(f"✅ 총 {len(df)}개 리뷰 저장 완료: {os.path.abspath(csv_filename)}")
    return df

# 앱 ID와 국가 설정
country = 'kr'
ios_app_id = '1469506430' #앱에 따라 ID 수정

# 실행 (최대 10페이지)
df_reviews = get_ios_reviews_all(ios_app_id, country, max_page=10)
print(df_reviews.head())


✅ 총 500개 리뷰 저장 완료: /content/app_reviews_1469506430_kr.csv
          작성자  별점                 제목  \
0         구방니   5        영원히썸원이열리지않음   
1         캥사이   2      광고 너무한 거 아닌가요   
2       sksbs   5     재밌어요 아쉬운점도 있고!   
3        마크대지   2  썸로그 좀 고쳐주세요 ㅠㅠㅠㅠㅠ   
4  Herrrrrr07   5                  .   

                                                  내용  \
0      백만년을해도열리지않음난영원히섬원을못하고영원히친구핰ㄴ테욕을머금도와주세요썸원안열려미친   
1  아니 100자 쓰는 것만 광고 보게 하더니 요즘은 안 넘어도 광고가 2개씩 나와요;...   
2                 질문팩에 질문들이 더 여러가지가 업데이트되었으면 좋겠어요 ㅋㅋ   
3  썸로그 업데이트 됐는데 이상해졌어요 \n지금까지 써놓은 썸로그를 우리함께로 한적이 ...   
4                     비록 헤어지긴 했지만 많은 추억들 만들어줘서 감사합니다   

                        작성일    앱버전  
0 2025-06-07 06:49:23-07:00  2.2.0  
1 2025-06-07 06:10:44-07:00  2.2.0  
2 2025-06-07 00:36:22-07:00  2.2.0  
3 2025-06-06 08:44:56-07:00  2.2.0  
4 2025-06-06 07:47:16-07:00  2.2.0  
